# 02 · Análisis Exploratorio de Datos (EDA) — Titanic

**Objetivo (Fase 2):** comprender la naturaleza de los datos del Titanic, evaluar su calidad, descubrir patrones y plantear hipótesis verificables.

**Contenido**
1. Carga y diccionario de datos
2. Calidad: duplicados y variables redundantes
3. Valores faltantes
4. Valores atípicos (outliers)
5. Distribuciones y análisis univariado
6. Análisis bivariado y multivariado
7. Hipótesis y su verificación estadística
8. Insights principales

Incluye controles de calidad verificables, intervalos de Wilson y ajuste de Holm, integrados en las secciones correspondientes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.precision", 3)
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/titanic.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecutar desde el repositorio o su carpeta notebooks.")
DATA = ROOT / "data"
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
OUT = ROOT / "reports" / "mejoras"
OUT.mkdir(parents=True, exist_ok=True)

def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()

df = pd.read_csv(DATA / "titanic.csv")
print(df.shape)
df.head()

(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.250,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.283,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.925,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.100,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.050,S,Third,man,True,NaN,Southampton,no,True


## 1. Diccionario de datos

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `survived` | Binaria (objetivo) | 1 = sobrevivió, 0 = no |
| `pclass` | Categórica ordinal | Clase del tiquete (1 = primera, 2 = segunda, 3 = tercera) |
| `sex` | Categórica nominal | Sexo del pasajero |
| `age` | Numérica continua | Edad en años |
| `sibsp` | Numérica discreta | N.º de hermanos/cónyuges a bordo |
| `parch` | Numérica discreta | N.º de padres/hijos a bordo |
| `fare` | Numérica continua | Tarifa pagada (libras esterlinas) |
| `embarked` | Categórica nominal | Puerto de embarque (C = Cherbourg, Q = Queenstown, S = Southampton) |
| `class` | Categórica | Igual a `pclass`, en texto |
| `who` | Categórica | man / woman / child |
| `adult_male` | Booleana | Hombre adulto |
| `deck` | Categórica | Cubierta del camarote (A–G) |
| `embark_town` | Categórica | Igual a `embarked`, en texto |
| `alive` | Categórica | Igual a `survived`, en texto (yes/no) |
| `alone` | Booleana | Viajaba sin familia |

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    str    
 8   class        891 non-null    str    
 9   who          891 non-null    str    
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    str    
 12  embark_town  889 non-null    str    
 13  alive        891 non-null    str    
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), str(7)
memory usage: 92.4 KB


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
survived,891.0,0.384,0.487,0.00,0.000,0.000,1.0,1.000
pclass,891.0,2.309,0.836,1.00,2.000,3.000,3.0,3.000
age,714.0,29.699,14.526,0.42,20.125,28.000,38.0,80.000
sibsp,891.0,0.523,1.103,0.00,0.000,0.000,1.0,8.000
parch,891.0,0.382,0.806,0.00,0.000,0.000,0.0,6.000
fare,891.0,32.204,49.693,0.00,7.910,14.454,31.0,512.329


In [4]:
df.select_dtypes(exclude="number").describe().T

,count,unique,top,freq
sex,891,2,male,577
embarked,889,3,S,644
class,891,3,Third,491
who,891,3,man,537
adult_male,891,2,True,537
deck,203,7,C,59
embark_town,889,3,Southampton,644
alive,891,2,no,549
alone,891,2,True,537


## 2. Calidad: duplicados y variables redundantes

In [5]:
print("Filas duplicadas exactas:", df.duplicated().sum())

pares = [("survived", "alive"), ("pclass", "class"), ("embarked", "embark_town")]
for a, b in pares:
    print(f"{a} vs {b}: relación uno a uno →", (df.groupby(a)[b].nunique(dropna=True) <= 1).all())

print("¿alone == (sibsp + parch == 0)?", (df["alone"] == ((df.sibsp + df.parch) == 0)).all())
print("¿adult_male == (who == 'man')?", (df["adult_male"] == (df.who == "man")).all())

Filas duplicadas exactas: 107
survived vs alive: relación uno a uno → True
pclass vs class: relación uno a uno → True
embarked vs embark_town: relación uno a uno → True
¿alone == (sibsp + parch == 0)? True
¿adult_male == (who == 'man')? True


**Hallazgo de calidad:** hay **107 filas idénticas posteriores a la primera**. Sin nombre, identificador ni número de tiquete, no podemos establecer si son registros repetidos o pasajeros diferentes con el mismo perfil. **Se conservan** para no eliminar observaciones sin evidencia suficiente.

Varias columnas son redundantes: `alive` reproduce `survived`, `class` representa `pclass`, `embark_town` representa `embarked`, `alone` se deriva de familiares y `adult_male` se deriva de `who`. `alive` no debe entrar como predictor porque revela el objetivo.

### 2.1 Controles de calidad y trazabilidad

Los controles revisan categorías permitidas, edades y conteos plausibles, y coherencia de columnas derivadas. Los valores ausentes se reportan por separado. Las filas idénticas se conservan: sin identificador no podemos asegurar que sean errores ni que sean personas distintas. Estos controles no certifican la exactitud histórica de cada registro.

In [6]:
reglas = {
    "objetivo binario y presente": df.survived.isin([0, 1]),
    "clase entre 1 y 3": df.pclass.isin([1, 2, 3]),
    "sexo documentado": df.sex.isin(["female", "male"]),
    "edad entre 0 y 100 o ausente": df.age.isna() | df.age.between(0, 100),
    "tarifa no negativa y presente": df.fare.ge(0),
    "sibsp entero no negativo": df.sibsp.ge(0) & df.sibsp.mod(1).eq(0),
    "parch entero no negativo": df.parch.ge(0) & df.parch.mod(1).eq(0),
    "puerto válido o ausente": df.embarked.isna() | df.embarked.isin(["C", "Q", "S"]),
    "alive coincide con objetivo": df.alive.map({"yes": 1, "no": 0}).eq(df.survived),
    "alone coincide con familiares": df.alone.eq(df.sibsp.add(df.parch).eq(0)),
}
calidad = pd.DataFrame([
    {"control": nombre, "incumplimientos": int((~valido.fillna(False)).sum())}
    for nombre, valido in reglas.items()
])
faltantes = pd.DataFrame({"n": df.isna().sum(), "porcentaje": df.isna().mean()*100})
calidad.to_csv(OUT / "controles_calidad.csv", index=False)
faltantes.to_csv(OUT / "faltantes.csv", index_label="variable")
display(calidad, faltantes.query("n > 0"))
print("Filas idénticas posteriores a la primera:", df.duplicated().sum())
assert calidad.incumplimientos.sum() == 0, "Revisar registros antes de continuar."

,control,incumplimientos
0,objetivo binario y presente,0
1,clase entre 1 y 3,0
2,sexo documentado,0
3,edad entre 0 y 100 o ausente,0
4,tarifa no negativa y presente,0
5,sibsp entero no negativo,0
6,parch entero no negativo,0
7,puerto válido o ausente,0
8,alive coincide con objetivo,0
9,alone coincide con familiares,0


,n,porcentaje
age,177,19.865
embarked,2,0.224
deck,688,77.217
embark_town,2,0.224


Filas idénticas posteriores a la primera: 107


### 2.2 Validaciones adicionales de dominio y coherencia

Los diez controles de la sección anterior siguen siendo la puerta de entrada: revisan que el objetivo, la clase, el sexo, la edad, la tarifa, los conteos familiares, el puerto, `alive` y `alone` caigan en el dominio permitido. Esta sección **no los reemplaza**. Añade comprobaciones de columnas derivadas, que en el Titanic de Seaborn repiten información con otro nombre o con una regla explícita.

La idea es separar tres preguntas distintas:

1. **Dominio.** ¿El valor está en el catálogo del diccionario? Por ejemplo, `who` solo puede ser `man`, `woman` o `child`, y `deck` solo A–G cuando el camarote está registrado.
2. **Coherencia entre columnas.** ¿Dos campos que deberían decir lo mismo realmente lo dicen? `class` debe traducir `pclass`, `embark_town` debe traducir `embarked`, y `adult_male` debe ser exactamente el caso `who == "man"`.
3. **Regla de construcción.** En esta versión de la base, `who` usa un corte de 16 años: con edad observada, un niño tiene menos de 16 y un adulto tiene 16 o más. Si la edad falta, no se juzga esa regla, porque no hay dato con qué contrastarla.

Un incumplimiento aquí sería un error de codificación o de carga, no un outlier. Por eso el criterio es binario: la fila cumple o no cumple. Las tarifas en cero se cuentan aparte. Hay pasajeros con `fare = 0` (tiquetes de cortesía o registros de tripulación en algunas versiones de la base) y eso puede ser plausible; no se mezcla con las validaciones de dominio.

Estas pruebas confirman consistencia interna. No certifican que cada edad o cada tarifa coincida con el registro histórico del naufragio.

In [ ]:
mapa_clase = {1: "First", 2: "Second", 3: "Third"}
mapa_puerto = {"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"}
edad_observada = df["age"].notna()

# Cada regla devuelve una Serie booleana: True = la fila cumple.
# No se reutiliza el diccionario `reglas` de la sección 2.1 para no alterar ese reporte.
validaciones = {
    "class coincide con pclass": df["class"].eq(df["pclass"].map(mapa_clase)),
    "who en man, woman o child": df["who"].isin(["man", "woman", "child"]),
    "edad observada coherente con who": (
        ~edad_observada
        | (df["who"].eq("child") & df["age"].lt(16))
        | (df["who"].ne("child") & df["age"].ge(16))
    ),
    "adult_male equivale a who = man": df["adult_male"].eq(df["who"].eq("man")),
    "who = man implica sexo masculino": df["who"].ne("man") | df["sex"].eq("male"),
    "who = woman implica sexo femenino": df["who"].ne("woman") | df["sex"].eq("female"),
    "alive solo yes o no": df["alive"].isin(["yes", "no"]),
    "deck en A-G o ausente": df["deck"].isna() | df["deck"].isin(list("ABCDEFG")),
    "puerto y ciudad faltan juntos": df["embarked"].isna().eq(df["embark_town"].isna()),
    "embark_town traduce embarked": (
        df["embarked"].isna() | df["embark_town"].eq(df["embarked"].map(mapa_puerto))
    ),
    "adult_male y alone son booleanos": (
        df["adult_male"].isin([True, False]) & df["alone"].isin([True, False])
    ),
    "conteos familiares en rango plausible": (
        df["sibsp"].between(0, 15) & df["parch"].between(0, 15)
    ),
}

criterios = {
    "class coincide con pclass": "1 = First, 2 = Second, 3 = Third",
    "who en man, woman o child": "catálogo cerrado del diccionario",
    "edad observada coherente con who": "niño < 16; adulto >= 16; edad ausente no se juzga",
    "adult_male equivale a who = man": "columna derivada, no una medición nueva",
    "who = man implica sexo masculino": "un niño puede ser de cualquier sexo",
    "who = woman implica sexo femenino": "woman no admite sexo masculino",
    "alive solo yes o no": "texto del objetivo; no se usa como predictor",
    "deck en A-G o ausente": "la ausencia se trata en la sección de faltantes",
    "puerto y ciudad faltan juntos": "embarked y embark_town son el mismo dato",
    "embark_town traduce embarked": "C/Q/S hacia Cherbourg, Queenstown y Southampton",
    "adult_male y alone son booleanos": "no deben llegar como texto libre",
    "conteos familiares en rango plausible": "sibsp y parch entre 0 y 15",
}

validaciones_extra = pd.DataFrame([
    {
        "control": nombre,
        "criterio": criterios[nombre],
        "incumplimientos": int((~cumple.fillna(False)).sum()),
    }
    for nombre, cumple in validaciones.items()
])
validaciones_extra.to_csv(OUT / "validaciones_adicionales.csv", index=False)

tarifas_cero = int(df["fare"].eq(0).sum())
print(f"Tarifas en cero (revisión, no incumplimiento de dominio): {tarifas_cero}")
print(
    "Se conservan: una tarifa nula puede ser un tiquete de cortesía "
    "y no demuestra por sí sola un error de captura."
)
display(validaciones_extra)
assert validaciones_extra["incumplimientos"].sum() == 0, (
    "Hay filas incoherentes con el diccionario. Revisar antes de interpretar el EDA."
)

**Lectura de estas validaciones.** Las doce reglas adicionales deben quedar en cero incumplimientos. Si alguna fila fallara, el `assert` detiene el notebook antes de interpretar faltantes, outliers o hipótesis: no tiene sentido describir patrones sobre columnas que se contradicen.

El corte de `who` solo se aplica cuando `age` está observada. Un niño puede ser hombre o mujer; eso no es un error. `adult_male` no aporta información nueva: es verdadero únicamente cuando `who` es `man`. `alive`, `class` y `embark_town` tampoco: reproducen `survived`, `pclass` y `embarked`. Por eso más adelante se tratan como redundantes y `alive` queda fuera de cualquier predictor, porque revelaría el objetivo.

Las tarifas en cero no entran en la tabla de incumplimientos. Se imprimen para no perderlas de vista, pero se conservan: una tarifa nula puede corresponder a un tiquete de cortesía y no alcanza para declararla error de captura. El archivo `reports/mejoras/validaciones_adicionales.csv` guarda este segundo reporte sin sobrescribir `controles_calidad.csv`.

## 3. Valores faltantes

In [7]:
faltantes = pd.DataFrame({
    "n_faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
}).query("n_faltantes > 0").sort_values("porcentaje", ascending=False)
faltantes

,n_faltantes,porcentaje
deck,688,77.22
age,177,19.87
embarked,2,0.22
embark_town,2,0.22


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [2, 1]})
sns.heatmap(df.isna(), cbar=False, cmap="rocket_r", yticklabels=False, ax=axes[0])
axes[0].set_title("Mapa de valores faltantes (franja oscura = dato ausente)")
faltantes["porcentaje"].plot(kind="bar", ax=axes[1], color="#c0392b")
axes[1].set_ylabel("% faltante"); axes[1].set_title("Porcentaje de faltantes por variable")
for i, v in enumerate(faltantes["porcentaje"]):
    axes[1].text(i, v + 1, f"{v}%", ha="center")
guardar("02_valores_faltantes.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


### ¿Los faltantes son aleatorios?

Si la ausencia depende de otras variables, eliminar filas sesgaría el análisis. Se compara la tasa de faltantes de `age` y `deck` por clase y por supervivencia.

In [9]:
df["age_faltante"] = df["age"].isna()
df["deck_faltante"] = df["deck"].isna()
print("% de faltantes por clase:")
print((df.groupby("pclass")[["age_faltante", "deck_faltante"]].mean() * 100).round(1))
print("\nSupervivencia según si 'deck' está registrado:")
print(df.groupby("deck_faltante")["survived"].mean().round(3))
print("\nSupervivencia según si 'age' está registrada:")
print(df.groupby("age_faltante")["survived"].mean().round(3))

% de faltantes por clase:
        age_faltante  deck_faltante
pclass                             
1               13.9           19.0
2                6.0           91.3
3               27.7           97.6

Supervivencia según si 'deck' está registrado:
deck_faltante
False    0.670
True     0.299
Name: survived, dtype: float64

Supervivencia según si 'age' está registrada:
age_faltante
False    0.406
True     0.294
Name: survived, dtype: float64


**Interpretación y estrategia de tratamiento:**

| Variable | % faltante | Evidencia observada | Estrategia propuesta |
|----------|-----------|--------------------|----------------------|
| `deck` | ≈77 % | La ausencia se asocia con clase y supervivencia. Esto no permite distinguir por sí solo MAR de MNAR. | Reemplazar por `tiene_cubierta`; imputar categorías con tanta ausencia tendría incertidumbre alta. |
| `age` | ≈20 % | La ausencia cambia según clase. El mecanismo no queda identificado. | Mediana por `pclass` y `sex`, más indicador `edad_faltante`. |
| `embarked` / `embark_town` | 0.2 % (2 filas) | Pocos casos; no basta para afirmar que sean aleatorios. | Imputar `embarked` con moda y retirar su columna redundante. |

Eliminar todas las filas con algún faltante dejaría 182 registros. La imputación es una decisión práctica que no recupera los valores verdaderos ni garantiza eliminar el sesgo.

In [10]:
print("Filas si se eliminara todo registro con algún faltante:", df.drop(columns=["age_faltante","deck_faltante"]).dropna().shape[0])

Filas si se eliminara todo registro con algún faltante: 182


## 4. Valores atípicos (outliers)

Se usan dos métodos estadísticos y un método gráfico:
- **IQR:** atípico si está por fuera de [Q1 − 1.5·IQR, Q3 + 1.5·IQR].
- **Z-score:** atípico si |z| > 3.

In [11]:
numericas = ["age", "fare", "sibsp", "parch"]

def outliers_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < li) | (s > ls)).sum(), li, ls

filas = []
for c in numericas:
    s = df[c].dropna()
    n_iqr, li, ls = outliers_iqr(s)
    n_z = (np.abs(stats.zscore(s)) > 3).sum()
    filas.append([c, round(li, 2), round(ls, 2), n_iqr, round(n_iqr / len(s) * 100, 1), n_z, s.max()])
tabla_out = pd.DataFrame(filas, columns=["variable", "límite_inf_IQR", "límite_sup_IQR",
                                         "outliers_IQR", "%_IQR", "outliers_z>3", "máximo"])
tabla_out

,variable,límite_inf_IQR,límite_sup_IQR,outliers_IQR,%_IQR,outliers_z>3,máximo
0,age,-6.69,64.81,11,1.5,2,80.000
1,fare,-26.72,65.63,116,13.0,20,512.329
2,sibsp,-1.50,2.50,46,5.2,30,8.000
3,parch,0.00,0.00,213,23.9,15,6.000


In [12]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, c in zip(axes, numericas):
    sns.boxplot(y=df[c], ax=ax, color="#5DADE2")
    ax.set_title(c)
plt.suptitle("Boxplots de las variables numéricas", y=1.02)
guardar("02_boxplots_outliers.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=df, x="pclass", y="fare", ax=axes[0], palette="Set2", hue="pclass", legend=False)
axes[0].set_title("Tarifa por clase: los outliers se concentran en 1.ª clase")
sns.scatterplot(data=df, x="age", y="fare", hue="survived", style="pclass", ax=axes[1], alpha=0.7)
axes[1].set_title("Edad vs tarifa (color = supervivencia)")
guardar("02_outliers_fare_por_clase.png")
print(df.sort_values("fare", ascending=False)[["pclass", "sex", "age", "fare", "survived"]].head(6))

     pclass     sex   age     fare  survived
258       1  female  35.0  512.329         1
737       1    male  35.0  512.329         1
679       1    male  36.0  512.329         1
88        1  female  23.0  263.000         1
27        1    male  19.0  263.000         0
341       1  female  24.0  263.000         1


/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


**Discusión: ¿conservar o ajustar?**

- **`fare`:** 116 observaciones señaladas por IQR. Las tarifas altas se concentran en primera clase y son plausibles; el boxplot no demuestra su exactitud histórica. El CSV no incluye tiquete, por lo que no permite confirmar si tarifas iguales pertenecen a un mismo tiquete. Se conservan y se aplica `log1p`. También se conservan 15 tarifas cero; su causa no está documentada aquí.
- **`age`:** las 11 edades señaladas por IQR son plausibles; se conservan.
- **`sibsp` y `parch`:** los conteos familiares altos pueden ser legítimos. En `parch`, Q1=Q3=0 hace que IQR marque todo valor positivo: 213 observaciones. Por tanto, `fare` no es la variable con más marcas y una marca estadística no equivale a error.

Decisión: conservar las observaciones y documentar las limitaciones de los detectores.

## 5. Distribuciones y análisis univariado

### 5.1 Variables numéricas

In [14]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, c in zip(axes.ravel(), numericas):
    sns.histplot(df[c].dropna(), kde=True, ax=ax, color="#2E86C1", bins=30)
    ax.axvline(df[c].mean(), color="red", ls="--", label=f"media={df[c].mean():.2f}")
    ax.axvline(df[c].median(), color="green", ls=":", label=f"mediana={df[c].median():.2f}")
    ax.set_title(f"{c} — asimetría = {df[c].skew():.2f}"); ax.legend()
guardar("02_histogramas_numericas.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


In [15]:
forma = pd.DataFrame({
    "media": df[numericas].mean(), "mediana": df[numericas].median(),
    "desv_std": df[numericas].std(), "asimetría": df[numericas].skew(),
    "curtosis": df[numericas].kurt(),
})
forma["p_valor_normalidad"] = [stats.normaltest(df[c].dropna()).pvalue for c in numericas]
forma.round(3)

,media,mediana,desv_std,asimetría,curtosis,p_valor_normalidad
age,29.699,28.000,14.526,0.389,0.178,0.0
fare,32.204,14.454,49.693,4.787,33.398,0.0
sibsp,0.523,0.000,1.103,3.695,17.880,0.0
parch,0.382,0.000,0.806,2.749,9.778,0.0


In [16]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df["fare"], kde=True, ax=axes[0], color="#E67E22")
axes[0].set_title(f"fare original — asimetría {df.fare.skew():.2f}")
sns.histplot(np.log1p(df["fare"]), kde=True, ax=axes[1], color="#27AE60")
axes[1].set_title(f"log(1 + fare) — asimetría {np.log1p(df.fare).skew():.2f}")
stats.probplot(np.log1p(df["fare"]), plot=axes[2]); axes[2].set_title("Q-Q plot de log(1 + fare)")
guardar("02_transformacion_log_fare.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


**Interpretación:**
- **`age`:** casi simétrica (asimetría ≈ 0.39), ligeramente sesgada a la derecha, con un pequeño pico de niños menores de 5 años. Media y mediana son cercanas (≈ 29.7 vs 28). Solo requiere **estandarización**.
- **`fare`:** **muy sesgada a la derecha** (asimetría ≈ 4.8, curtosis ≈ 33). La media (≈ 32) es más del doble de la mediana (≈ 14.5). La transformación `log1p` reduce la asimetría a ≈ 0.4, por lo que **se aplicará en el preprocesamiento**.
- **`sibsp` y `parch`:** discretas y concentradas en 0 (la mayoría viajaba sin esos familiares).
- Ninguna variable es normal (p-valor < 0.05 en la prueba de D'Agostino), lo que justifica usar **medianas** para imputar y **pruebas no paramétricas** para comparar grupos.

### 5.2 Variables categóricas

In [17]:
categoricas = ["survived", "pclass", "sex", "embarked", "who", "deck", "alone"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), categoricas):
    conteo = df[c].astype(object).fillna("N/D").astype(str).value_counts().sort_index()
    colores = ["#c0392b" if k == "N/D" else sns.color_palette("Set2")[i % 8] for i, k in enumerate(conteo.index)]
    ax.bar(conteo.index, conteo.values, color=colores)
    for i, v in enumerate(conteo.values):
        ax.text(i, v, f"{v/len(df)*100:.0f}%", ha="center", va="bottom", fontsize=9)
    ax.set_title(c); ax.set_ylabel("frecuencia")
axes.ravel()[-1].axis("off")
guardar("02_barras_categoricas.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


In [18]:
for c in ["survived", "pclass", "sex", "embarked"]:
    print(f"--- {c} ---")
    print(df[c].value_counts(normalize=True).mul(100).round(1).to_string(), "\n")

--- survived ---
survived
0    61.6
1    38.4 

--- pclass ---
pclass
3    55.1
1    24.2
2    20.7 

--- sex ---
sex
male      64.8
female    35.2 

--- embarked ---
embarked
S    72.4
C    18.9
Q     8.7 



**Interpretación:**
- **Clase objetivo desbalanceada:** solo ≈ 38 % sobrevivió (342 de 891). Este desbalance moderado debe tenerse en cuenta al modelar (por ejemplo, usar métricas como F1 o validación estratificada).
- **Predominio de 3.ª clase** (≈ 55 %) y de **hombres** (≈ 65 %).
- **Southampton** fue el puerto de la gran mayoría (≈ 72 %).
- En `deck`, la categoría más frecuente es "NaN", lo que confirma que no conviene usarla directamente.

## 6. Análisis bivariado y multivariado

### 6.1 Supervivencia frente a variables categóricas (tablas cruzadas)

In [19]:
for c in ["sex", "pclass", "embarked", "who"]:
    print(f"\n=== Tasa de supervivencia por {c} ===")
    print(pd.crosstab(df[c], df["survived"], margins=True, normalize="index").round(3))


=== Tasa de supervivencia por sex ===
survived      0      1
sex                   
female    0.258  0.742
male      0.811  0.189
All       0.616  0.384

=== Tasa de supervivencia por pclass ===
survived      0      1
pclass                
1         0.370  0.630
2         0.527  0.473
3         0.758  0.242
All       0.616  0.384

=== Tasa de supervivencia por embarked ===
survived      0      1
embarked              
C         0.446  0.554
Q         0.610  0.390
S         0.663  0.337
All       0.618  0.382

=== Tasa de supervivencia por who ===
survived      0      1
who                   
child     0.410  0.590
man       0.836  0.164
woman     0.244  0.756
All       0.616  0.384


In [20]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, c in zip(axes, ["sex", "pclass", "embarked", "who"]):
    sns.barplot(data=df, x=c, y="survived", ax=ax, hue=c, legend=False, palette="Set2", errorbar=("ci", 95))
    ax.set_ylim(0, 1); ax.set_ylabel("tasa de supervivencia"); ax.set_title(f"Supervivencia por {c}")
    ax.axhline(df.survived.mean(), color="gray", ls="--", lw=1)
guardar("02_supervivencia_por_categoricas.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


In [21]:
tabla = pd.crosstab([df["pclass"], df["sex"]], df["survived"], normalize="index")[1].unstack()
plt.figure(figsize=(6, 4))
sns.heatmap(tabla, annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1, cbar_kws={"label": "tasa de supervivencia"})
plt.title("Supervivencia por clase y sexo (interacción)")
guardar("02_heatmap_clase_sexo.png")
tabla.round(3)

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


sex,female,male
pclass,,
1,0.968,0.369
2,0.921,0.157
3,0.500,0.135


**Interpretación:** el sexo y la clase son los factores más fuertes. Las **mujeres de 1.ª y 2.ª clase sobrevivieron más del 90 %**, mientras que los **hombres de 2.ª y 3.ª clase sobrevivieron menos del 16 %**. Además hay **interacción**: el efecto de la clase es mucho mayor en las mujeres (de ≈ 97 % a ≈ 50 %) que en los hombres. El puerto de Cherbourg muestra mayor supervivencia, pero probablemente porque allí embarcaron más pasajeros de 1.ª clase (se verifica abajo).

### 6.1.1 Tasas con tamaño de muestra e intervalos

Una tasa debe acompañarse del número de observaciones. Se calculan intervalos de Wilson del 95 % por sexo y clase. Son intervalos descriptivos bajo un supuesto binomial de independencia; familiares pueden estar relacionados y esta muestra histórica no representa una selección aleatoria de cualquier población. Un intervalo no mide el efecto causal de sexo o clase.

In [22]:
tasas = (df.groupby(["pclass", "sex"], observed=True).survived
         .agg(n="size", sobrevivientes="sum").reset_index())
tasas["tasa"] = tasas.sobrevivientes / tasas.n
intervalos = [stats.binomtest(int(k), int(n)).proportion_ci(
    confidence_level=0.95, method="wilson")
    for k, n in zip(tasas.sobrevivientes, tasas.n)]
tasas["ic95_inf"] = [ic.low for ic in intervalos]
tasas["ic95_sup"] = [ic.high for ic in intervalos]
tasas.to_csv(OUT / "supervivencia_intervalos.csv", index=False)
display(tasas.round(4))
fig, ax = plt.subplots(figsize=(9, 4.5))
pos = np.arange(len(tasas))
ax.errorbar(tasas.tasa*100, pos,
    xerr=np.vstack([tasas.tasa-tasas.ic95_inf, tasas.ic95_sup-tasas.tasa])*100,
    fmt="o", capsize=5, color="#176B87")
sexo = tasas.sex.map({"female": "mujeres", "male": "hombres"})
ax.set_yticks(pos, [f"Clase {c}, {s} (n={n})" for c, s, n in zip(tasas.pclass, sexo, tasas.n)])
ax.set_xlim(0, 100)
ax.set_xlabel("Supervivencia (%) e intervalo de Wilson del 95 %")
ax.set_title("La tasa y su incertidumbre dependen del grupo")
ax.grid(axis="x", alpha=.25)
fig.tight_layout()
fig.savefig(FIG / "02_supervivencia_intervalos.png", dpi=140)
plt.show()

,pclass,sex,n,sobrevivientes,tasa,ic95_inf,ic95_sup
0,1,female,94,91,0.968,0.910,0.989
1,1,male,122,45,0.369,0.288,0.457
2,2,female,76,70,0.921,0.838,0.963
3,2,male,108,17,0.157,0.101,0.238
4,3,female,144,72,0.500,0.419,0.581
5,3,male,347,47,0.135,0.103,0.175


/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/3169650086.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
print("Composición de clase por puerto de embarque (%):")
pd.crosstab(df["embarked"], df["pclass"], normalize="index").mul(100).round(1)

Composición de clase por puerto de embarque (%):


pclass,1,2,3
embarked,,,
C,50.6,10.1,39.3
Q,2.6,3.9,93.5
S,19.7,25.5,54.8


### 6.2 Supervivencia frente a variables numéricas

In [24]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.kdeplot(data=df, x="age", hue="survived", fill=True, common_norm=False, ax=axes[0], alpha=0.4)
axes[0].set_title("Densidad de edad según supervivencia")
sns.boxplot(data=df, x="survived", y="fare", ax=axes[1], hue="survived", legend=False, palette="Set2")
axes[1].set_yscale("log"); axes[1].set_title("Tarifa (escala log) según supervivencia")
df["tamano_familia"] = df["sibsp"] + df["parch"] + 1
sns.barplot(data=df, x="tamano_familia", y="survived", ax=axes[2], color="#8E44AD", errorbar=None)
axes[2].set_title("Supervivencia por tamaño de familia")
guardar("02_supervivencia_por_numericas.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


In [25]:
df["grupo_edad"] = pd.cut(df["age"], bins=[0, 12, 18, 30, 45, 60, 80],
                          labels=["0-12", "13-18", "19-30", "31-45", "46-60", "61-80"])
pd.crosstab(df["grupo_edad"], df["survived"], normalize="index", margins=True).round(3)

survived,0,1
grupo_edad,,
0-12,0.420,0.580
13-18,0.571,0.429
19-30,0.644,0.356
31-45,0.574,0.426
46-60,0.593,0.407
61-80,0.773,0.227
All,0.594,0.406


**Interpretación:**
- Los **niños de 12 años o menos** tuvieron una supervivencia notablemente mayor (≈ 58 %); la baja correlación lineal global no descarta relaciones por edad.
- Los sobrevivientes pagaron **tarifas mayores** (mediana ≈ 26 vs ≈ 10), lo que es compatible con la asociación entre tarifa y clase.
- La relación con el tamaño de familia es **no lineal**: viajar solo (≈ 30 %) o en familias grandes (5 o más, ≈ 16 % en conjunto) se asocia con menor supervivencia; las familias de 2 a 4 personas sobrevivieron más (≈ 55–72 %, ≈ 58 % en conjunto).

### 6.3 Correlaciones y mapa de calor

In [26]:
df_corr = df[["survived", "pclass", "age", "sibsp", "parch", "fare", "tamano_familia"]].copy()
df_corr["sex_female"] = (df["sex"] == "female").astype(int)
df_corr["log_fare"] = np.log1p(df["fare"])
df_corr["tiene_cubierta"] = df["deck"].notna().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
sns.heatmap(df_corr.corr(method="pearson"), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0], square=True)
axes[0].set_title("Correlación de Pearson")
sns.heatmap(df_corr.corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1], square=True)
axes[1].set_title("Correlación de Spearman (robusta a outliers)")
guardar("02_heatmap_correlaciones.png")

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/1404552930.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / nombre, dpi=120, bbox_inches="tight"); plt.show()


In [27]:
df_corr.corr()["survived"].drop("survived").sort_values(key=abs, ascending=False).round(3)

sex_female        0.543
pclass           -0.338
log_fare          0.330
tiene_cubierta    0.320
fare              0.257
parch             0.082
age              -0.077
sibsp            -0.035
tamano_familia    0.017
Name: survived, dtype: float64

**Interpretación:**
- La variable más asociada con sobrevivir es el **sexo femenino** (r ≈ 0.54), seguida de la **clase** (r ≈ −0.34: a mayor número de clase, menor supervivencia) y de **`tiene_cubierta`** (r ≈ 0.32), lo que confirma que el faltante de `deck` es informativo.
- **`pclass` y `fare` están fuertemente correlacionadas** de forma negativa (≈ −0.55 en Pearson, ≈ −0.69 en Spearman). Esto indica redundancia parcial, lo que **justifica aplicar PCA** en la siguiente fase.
- `sibsp`, `parch` y `tamano_familia` están muy correlacionadas entre sí (multicolinealidad).
- La edad tiene correlación lineal muy baja con la supervivencia (≈ −0.08), pero ya vimos que su efecto **no es lineal** (los niños).

### 6.4 Pairplot (vista multivariada)

In [28]:
g = sns.pairplot(df[["survived", "age", "fare", "pclass", "tamano_familia"]].assign(fare=np.log1p(df.fare)).rename(columns={"fare": "log_fare"}),
                 hue="survived", diag_kind="kde", plot_kws={"alpha": 0.5, "s": 20}, height=2.3)
g.figure.suptitle("Pairplot de variables numéricas por supervivencia", y=1.02)
plt.savefig(FIG / "02_pairplot.png", dpi=110, bbox_inches="tight"); plt.show()

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58028/3179249612.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(FIG / "02_pairplot.png", dpi=110, bbox_inches="tight"); plt.show()


## 7. Hipótesis y ajuste por comparaciones múltiples

Se contrastan cinco hipótesis del EDA: supervivencia frente a sexo, clase, tarifa, edad ≤12 y tamaño de familia y se ajustan sus p-valores con Holm. En chi-cuadrado se informa la menor frecuencia esperada; se conserva la corrección de continuidad predeterminada en tablas 2×2 para mantener la misma convención del análisis inicial. Para V de Cramér se usa el estadístico de Pearson sin esa corrección. Para Mann–Whitney se añade una correlación biserial de rangos, orientada a tarifas mayores entre sobrevivientes.

Holm controla el error familiar de estas cinco pruebas si sus p-valores son válidos. No elimina el sesgo de haber formulado hipótesis después de mirar los datos ni la posible dependencia entre familiares. H4 usa solo edades observadas y compara ≤12 con >12, no exclusivamente con adultos. Mann–Whitney compara distribuciones/rangos; no es, en general, una prueba solo de medianas.

In [29]:
filas = []
def agregar_chi(nombre, grupos, objetivo):
    tabla = pd.crosstab(grupos, objetivo)
    chi2, p, gl, esperadas = stats.chi2_contingency(tabla)
    pearson = stats.chi2_contingency(tabla, correction=False).statistic
    n = int(tabla.to_numpy().sum())
    v = np.sqrt(pearson / (n * (min(tabla.shape)-1)))
    filas.append(dict(hipotesis=nombre, prueba="Chi-cuadrado", n=n,
                      estadistico=chi2, p=p, esperada_min=esperadas.min(),
                      medida_efecto="V de Cramér", efecto=v))
agregar_chi("H1 sexo", df.sex, df.survived)
agregar_chi("H2 clase", df.pclass, df.survived)
a = df.loc[df.survived.eq(1), "fare"]
b = df.loc[df.survived.eq(0), "fare"]
u, p = stats.mannwhitneyu(a, b, alternative="greater")
filas.append(dict(hipotesis="H3 tarifa", prueba="Mann–Whitney", n=len(a)+len(b),
                  estadistico=u, p=p, esperada_min=np.nan,
                  medida_efecto="Biserial de rangos", efecto=2*u/(len(a)*len(b))-1))
edad = df.dropna(subset=["age"])
agregar_chi("H4 edad ≤12", edad.age.le(12), edad.survived)
familia = pd.cut(df.sibsp + df.parch + 1, [0, 1, 4, np.inf], labels=["solo", "2–4", "5+"])
agregar_chi("H5 familia", familia, df.survived)
hipotesis = pd.DataFrame(filas)
p_original = hipotesis.p.to_numpy()
orden = np.argsort(p_original)
p_ajustado = np.empty_like(p_original)
p_ajustado[orden] = np.minimum(1, np.maximum.accumulate(
    p_original[orden] * np.arange(len(orden), 0, -1)))
hipotesis["p_holm"] = p_ajustado
hipotesis["rechaza_h0_holm_005"] = p_ajustado < .05
hipotesis.to_csv(OUT / "hipotesis_holm.csv", index=False)
display(hipotesis)
print("Pruebas significativas tras Holm:", hipotesis.rechaza_h0_holm_005.sum(), "de", len(hipotesis))

,hipotesis,prueba,n,estadistico,p,esperada_min,medida_efecto,efecto,p_holm,rechaza_h0_holm_005
0,H1 sexo,Chi-cuadrado,891,260.717,1.197e-58,120.525,V de Cramér,0.543,5.987e-58,True
1,H2 clase,Chi-cuadrado,891,102.889,4.549e-23,70.626,V de Cramér,0.340,1.820e-22,True
2,H3 tarifa,Mann–Whitney,891,129951.500,2.277e-22,NaN,Biserial de rangos,0.384,6.830e-22,True
3,H4 edad ≤12,Chi-cuadrado,714,8.758,3.082e-03,28.025,V de Cramér,0.116,3.082e-03,True
4,H5 familia,Chi-cuadrado,891,74.537,6.523e-17,23.798,V de Cramér,0.289,1.305e-16,True


Pruebas significativas tras Holm: 5 de 5


**Conclusiones:** las cinco pruebas siguen con p < 0.05 tras el ajuste de Holm. El mayor p ajustado es aproximadamente 0.00308 y corresponde al contraste de edad, realizado con 714 edades observadas. H1 presenta V de Cramér ≈0.54 y H2 ≈0.34.

Se trata de asociaciones exploratorias. Los intervalos y p-valores dependen de supuestos de independencia; el parentesco puede afectarlos. Holm no corrige haber formulado hipótesis a partir de esta misma muestra ni demuestra causas históricas.

## 8. Insights principales

**Patrones relevantes encontrados**
1. **El sexo presenta una asociación bivariada fuerte:** las mujeres sobrevivieron ≈ 74 % frente a ≈ 19 % de los hombres.
2. **La clase social marcó la diferencia:** la supervivencia cae de ≈ 63 % en 1.ª clase a ≈ 47 % en 2.ª y ≈ 24 % en 3.ª.
3. **Sexo y clase interactúan:** las mujeres de 1.ª y 2.ª clase sobrevivieron más del 90 %, pero en 3.ª clase solo el 50 %; los hombres de 2.ª y 3.ª clase, menos del 16 %.
4. **Los niños de 12 años o menos sobrevivieron más** (≈ 58 %), sin que esto permita descartar relaciones de edad en otros grupos.
5. **Viajar en familia mediana (2–4 personas) se asoció con mayor supervivencia** (≈ 58 %); viajar solo (≈ 30 %) o en familias de 5 o más (≈ 16 %) se asoció con una tasa menor (relación no lineal).
6. **Pagar más se asocia a sobrevivir**, y la tarifa también está asociada con la clase (correlación `pclass`–`fare` ≈ −0.55 a −0.69).
7. **Tener la cubierta registrada se asocia a mayor supervivencia** (≈ 67 % vs ≈ 30 %): el dato faltante es informativo por sí mismo.
8. **El puerto de Cherbourg tiene mayor supervivencia** y la mitad de sus pasajeros eran de 1.ª clase; la composición es una explicación posible que requiere análisis ajustado.

**Problemas de calidad detectados**
- `deck` con ≈ 77 % de faltantes (ausencia asociada a variables observadas) → indicador binario.
- `age` con ≈ 20 % de faltantes, más frecuentes en 3.ª clase → imputación por mediana agrupada + indicador.
- `embarked` con 2 faltantes → moda.
- Cinco columnas redundantes (`alive`, `class`, `embark_town`, `alone`, `adult_male`); `alive` causaría **fuga de información** si se usara para modelar.
- `fare` fuertemente sesgada con valores extremos plausibles → transformación logarítmica.
- Filas idénticas cuya identidad no puede resolverse sin identificadores (se conservan).
- Clase objetivo moderadamente desbalanceada (38 % / 62 %).

**Líneas de análisis para fases posteriores**
- Modelos de clasificación (regresión logística, árboles de decisión, random forest) para predecir `survived`, usando validación estratificada.
- Ingeniería de variables: `tamano_familia` en categorías, interacción `sexo × clase`, grupos de edad.
- Comparar el rendimiento con y sin las variables imputadas para medir el impacto del tratamiento de faltantes.

➡️ Siguiente paso: `03_preprocesamiento_pca.ipynb`.